In [ ]:
using SparseArrays

In [ ]:

function sparse_second_derivative_matrix(n, h)
    @assert n >= 1 "n must be at least 1"
    @assert h > 0 "h must be positive"

    main = fill(-2.0, n)
    off  = fill(1.0, n - 1)

    return (1 / h^2) * spdiagm(-1 => off, 0 => main, 1 => off)
end

In [ ]:
sparse_second_derivative_matrix(10,1)

In [ ]:
function heat!(du, u, p, t)
    alpha = p[1];
    dx = p[2];
    x = p[3];
    ga = p[4];
    gb = p[5];
    f = p[6];
    Dxx = p[7];
    
    du .= alpha * Dxx * u + f.(x,t);
    # apply boundary conditions
    du[1] += alpha/dx^2 * ga(t);
    du[end] += alpha/dx^2 * gb(t);
    
    du
end


In [ ]:
x = LinRange(-5.0, 5.0, 51)[2:end-1];


u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 5.0) # (t0, tmax)

alpha = 1;  # diffusivity
dx = x[2] - x[1];
n = length(x);
ga = t-> 0.01 * t;
gb = t-> 0;
f = (x,t) -> 0; # source/sink
Dxx = sparse_second_derivative_matrix(n, dx);

p = (alpha, dx, x, ga, gb, f, Dxx);

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
sol = solve(prob);

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 51);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 1)
end

In [ ]:
gif(anim, fps=15)